In [9]:
import numpy as np
import glob
import os
results_dir = '/projectnb/modislc/users/danc/SOFM/sofm_sim_study_fixphi/'
file_pattern = os.path.join(results_dir, "*.csv")
files = glob.glob(file_pattern)
print(f"Found {len(files)} result files. Loading...")
data_list = [np.atleast_2d(np.loadtxt(f, delimiter=',')) for f in files]
final_results = np.concatenate(data_list, axis=0)

Found 2400 result files. Loading...


In [10]:
final_results

array([[1.00000000e+04, 5.00000000e+00, 2.00000000e+01, ...,
        2.76602185e+00, 1.83980162e+00, 0.00000000e+00],
       [5.00000000e+02, 1.00000000e+01, 2.00000000e+00, ...,
        2.12534865e+00, 1.98218406e+00, 0.00000000e+00],
       [2.00000000e+02, 5.00000000e+00, 2.00000000e+00, ...,
        4.62274144e+00, 3.99887941e+00, 0.00000000e+00],
       ...,
       [5.00000000e+02, 5.00000000e+00, 2.00000000e+01, ...,
        7.89769835e+00, 6.90949846e+00, 0.00000000e+00],
       [2.00000000e+02, 5.00000000e+00, 2.00000000e+01, ...,
        1.82944015e+00, 2.81644341e+00, 0.00000000e+00],
       [5.00000000e+03, 5.00000000e+00, 2.00000000e+00, ...,
        1.79823923e+00, 1.64081340e+00, 0.00000000e+00]], shape=(2400, 12))

In [11]:
import numpy as np
import glob
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
def plot_consistency(df_results, sigma, prior_cov, max_lag, growing):
    """Generates boxplots for U, L, and Sigma losses (growing=False)."""
    titlesize = 20
    labelsize = 20
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Plot U Loss
    sns.boxplot(data=df_results, x='Sample_Size', y='U_MSE', ax=axes[0], color='blue', showfliers=False)
    sns.stripplot(data=df_results, x='Sample_Size', y='U_MSE', ax=axes[0], color='black', alpha=0.5, size=4)
    axes[0].set_title('Loadings (U)', fontsize=titlesize)
    axes[0].set_ylabel('Mean squared error', fontsize=labelsize)
    axes[0].set_xlabel('Sample size', fontsize=labelsize)

    # Plot L Loss
    sns.boxplot(data=df_results, x='Sample_Size', y='L_MSE', ax=axes[1], color='blue', showfliers=False)
    sns.stripplot(data=df_results, x='Sample_Size', y='L_MSE', ax=axes[1], color='black', alpha=0.5, size=4)
    axes[1].set_title('Singular values (L)', fontsize=titlesize)
    axes[1].set_ylabel('Mean squared error', fontsize=labelsize)
    axes[1].set_xlabel('Sample size', fontsize=labelsize)

    # Plot Sigma Loss
    sns.boxplot(data=df_results, x='Sample_Size', y='Sigma_MSE', ax=axes[2], color='blue', showfliers=False)
    sns.stripplot(data=df_results, x='Sample_Size', y='Sigma_MSE', ax=axes[2], color='black', alpha=0.5, size=4)
    axes[2].set_title('Noise singular value (sigma)', fontsize=titlesize)
    axes[2].set_ylabel('Squared error', fontsize=labelsize)
    axes[2].set_xlabel('Sample size', fontsize=labelsize)

    plt.tight_layout()
    plt.savefig(f"/projectnb/modislc/users/danc/SOFM/plots5/sofm_ULS_plots__sigma{sigma}_maxlag{max_lag}_growing{growing}_priorcov{prior_cov}.png", dpi=300)
    plt.close() # Use plt.close() instead of plt.show() to prevent notebook/terminal flooding


In [13]:
def plot_latlonrot(df_results, sigma, prior_cov, max_lag, growing):
    """Generates boxplots for lat, lon, and rot MSEs (growing=True)."""
    titlesize = 20
    labelsize = 20
    sns.set_theme(style="whitegrid")
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Plot lat Loss
    sns.boxplot(data=df_results, x='Sample_Size', y='lat_MSE', ax=axes[0], color='blue', showfliers=False)
    sns.stripplot(data=df_results, x='Sample_Size', y='lat_MSE', ax=axes[0], color='black', alpha=0.5, size=4)
    axes[0].set_title('Latitudinal length scales', fontsize=titlesize)
    axes[0].set_ylabel('Mean squared error', fontsize=labelsize)
    axes[0].set_xlabel('Sample size', fontsize=labelsize)

    # Plot lon Loss
    sns.boxplot(data=df_results, x='Sample_Size', y='lon_MSE', ax=axes[1], color='blue', showfliers=False)
    sns.stripplot(data=df_results, x='Sample_Size', y='lon_MSE', ax=axes[1], color='black', alpha=0.5, size=4)
    axes[1].set_title('Longitudinal length scales', fontsize=titlesize)
    axes[1].set_ylabel('Mean squared error', fontsize=labelsize)
    axes[1].set_xlabel('Sample size', fontsize=labelsize)

    # Plot rot Loss
    sns.boxplot(data=df_results, x='Sample_Size', y='rot_MSE', ax=axes[2], color='blue', showfliers=False)
    sns.stripplot(data=df_results, x='Sample_Size', y='rot_MSE', ax=axes[2], color='black', alpha=0.5, size=4)
    axes[2].set_title('Rotations', fontsize=titlesize)
    axes[2].set_ylabel('Squared error', fontsize=labelsize)
    axes[2].set_xlabel('Sample size', fontsize=labelsize)

    plt.tight_layout()
    plt.savefig(f"/projectnb/modislc/users/danc/SOFM/plots5/sofm_latlonrot_plots__sigma{sigma}_maxlag{max_lag}_growing{growing}_priorcov{prior_cov}.png", dpi=300)
    plt.close()


In [15]:
# 1. Load the data
file_pattern = os.path.join(results_dir, "*.csv")
files = glob.glob(file_pattern)

print(f"Found {len(files)} result files. Loading...")
data_list = [np.atleast_2d(np.loadtxt(f, delimiter=',')) for f in files]
final_results = np.concatenate(data_list, axis=0)

# 2. Convert to a Pandas DataFrame
# These column names map exactly to the array you generated:
# [n_samples, sigma, max_lag, prior_cov, growing, replicate, U_loss, L_loss, sigma_loss, lat_mse, lon_mse, rot_mse]
column_names = [
    'Sample_Size', 'Sigma', 'Max_Lag', 'Prior_Cov', 'Growing', 'Replicate',
    'U_MSE', 'L_MSE', 'Sigma_MSE', 'lat_MSE', 'lon_MSE', 'rot_MSE'
]
df = pd.DataFrame(final_results, columns=column_names)

# Convert Growing to boolean for cleaner logic
df['Growing'] = df['Growing'].astype(bool)

# Cast these to ints/floats to ensure seaborn handles categorical vs continuous correctly
df['Sample_Size'] = df['Sample_Size'].astype(int)
df['Sigma'] = df['Sigma'].astype(float)
df['Max_Lag'] = df['Max_Lag'].astype(int)
df['Prior_Cov'] = df['Prior_Cov'].astype(int)

initial_len = len(df)
df = df[df['lat_MSE'] != -999]
df = df[df['U_MSE'] != -999]
dropped_count = initial_len - len(df)
if dropped_count > 0:
    print(f"Removed {dropped_count} row(s) containing -999 error flags.")


# 3. Group the dataframe by the combination of parameters and plot
# groupby effectively handles the nested for-loops for you
grouped = df.groupby(['Sigma', 'Max_Lag', 'Prior_Cov', 'Growing'])

print("Generating plots...")
for (sigma, max_lag, prior_cov, growing), group_df in grouped:
    # Check if the dataframe partition has rows just in case
    if group_df.empty:
        continue
        
    print(f"Plotting for: Sigma={sigma}, Max_Lag={max_lag}, Prior_Cov={prior_cov}, Growing={growing}")
    
    # # Route to the correct plotting function based on 'growing' state
    # if growing:
    #     plot_latlonrot(group_df, sigma, prior_cov, max_lag, growing)
    # else:
    plot_consistency(group_df, sigma, prior_cov, max_lag, growing)

print("All plots saved successfully.")

Found 2400 result files. Loading...
Removed 4 row(s) containing -999 error flags.
Generating plots...
Plotting for: Sigma=5.0, Max_Lag=2, Prior_Cov=0, Growing=False
Plotting for: Sigma=5.0, Max_Lag=2, Prior_Cov=0, Growing=True
Plotting for: Sigma=5.0, Max_Lag=2, Prior_Cov=1, Growing=False
Plotting for: Sigma=5.0, Max_Lag=2, Prior_Cov=1, Growing=True
Plotting for: Sigma=5.0, Max_Lag=2, Prior_Cov=2, Growing=False
Plotting for: Sigma=5.0, Max_Lag=2, Prior_Cov=2, Growing=True
Plotting for: Sigma=5.0, Max_Lag=20, Prior_Cov=0, Growing=False
Plotting for: Sigma=5.0, Max_Lag=20, Prior_Cov=0, Growing=True
Plotting for: Sigma=5.0, Max_Lag=20, Prior_Cov=1, Growing=False
Plotting for: Sigma=5.0, Max_Lag=20, Prior_Cov=1, Growing=True
Plotting for: Sigma=5.0, Max_Lag=20, Prior_Cov=2, Growing=False
Plotting for: Sigma=5.0, Max_Lag=20, Prior_Cov=2, Growing=True
Plotting for: Sigma=10.0, Max_Lag=2, Prior_Cov=0, Growing=False
Plotting for: Sigma=10.0, Max_Lag=2, Prior_Cov=0, Growing=True
Plotting for: S